In [1]:
from icecube import dataio, icetray
import pandas as pd
from icecube import LeptonInjector # for EventProperties
from icecube import simclasses # for MMCTrackList



In [6]:
path = "/project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator/33939527/gen_001.i3.zst"
data_file = dataio.I3File(path)

In [10]:
data_file.pop_frame() 
data_file.pop_frame() 
frame = data_file.pop_frame() 
print(frame)

[ I3Frame  (DAQ):
  'Accepted_PulseMap' [DAQ] ==> I3Map<OMKey, vector<I3RecoPulse> > (45460)
  'EventProperties' [DAQ] ==> LeptonInjector::BasicEventProperties (140)
  'I3EventHeader' [DAQ] ==> I3EventHeader (99)
  'I3MCTree' [DAQ] ==> TreeBase::Tree<I3Particle, I3ParticleID, i3hash<I3ParticleID> > (6506)
  'I3MCTree_RNGState' [DAQ] ==> I3GSLRandomServiceState (87)
  'I3Photons' [DAQ] ==> I3Map<ModuleKey, I3Vector<I3CompressedPhoton> > (3005899)
  'LeptonInjectorProperties' [Simulation] ==> LeptonInjector::RangedInjectionConfiguration (2376162)
  'MMCTrackList' [DAQ] ==> I3Vector<I3MMCTrack> (304)
  'Noise_Dark' [DAQ] ==> I3Map<OMKey, vector<I3MCPE> > (566454)
  'Noise_K40' [DAQ] ==> I3Map<OMKey, vector<I3MCPE> > (1759083)
]



In [12]:
print(frame['EventProperties'])

[ BasicEventProperties 
         TotalEnergy : 2953.8
              Zenith : 2.00897
             Azimuth : 4.64901
         FinalStateX : 0.0822134
         FinalStateY : 0.544104
          FinalType1 : MuMinus
          FinalType2 : Hadrons
         InitialType : NuMu
                   X : 354.135
                   Y : -1941.87
                   Z : -634.68
    TotalColumnDepth : 938122
              Radius : 1973.89
     ImpactParameter : 532.556
]


In [13]:
print(frame['LeptonInjectorProperties'])

In [14]:
props = frame['LeptonInjectorProperties']
print(dir(props))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__instance_size__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'azimuthMaximum', 'azimuthMinimum', 'endcapLength', 'energyMaximum', 'energyMinimum', 'events', 'finalType1', 'finalType2', 'injectionRadius', 'powerlawIndex', 'zenithMaximum', 'zenithMinimum']


In [15]:
props = frame['LeptonInjectorProperties']

print('injectionRadius:', props.injectionRadius)
print('endcapLength:', props.endcapLength)
print('energyMinimum:', props.energyMinimum)
print('energyMaximum:', props.energyMaximum)
print('zenithMinimum:', props.zenithMinimum)
print('zenithMaximum:', props.zenithMaximum)
print('azimuthMinimum:', props.azimuthMinimum)
print('azimuthMaximum:', props.azimuthMaximum)
print('powerlawIndex:', props.powerlawIndex)
print('events:', props.events)
print('finalType1:', props.finalType1)
print('finalType2:', props.finalType2)

injectionRadius: 1000.0
endcapLength: 1100.0
energyMinimum: 100.0
energyMaximum: 1000000.0
zenithMinimum: 0.0
zenithMaximum: 3.141592653589793
azimuthMinimum: 0.0
azimuthMaximum: 6.283185307179586
powerlawIndex: 1.5
events: 100
finalType1: MuMinus
finalType2: Hadrons


In [17]:
from icecube import LeptonWeighter

ImportError: cannot import name 'LeptonWeighter' from 'icecube' (unknown location)

In [ ]:
import os
import glob
from icecube import icetray, dataio, dataclasses
import pandas as pd

def get_all_zst_files(base_path):
    pattern = os.path.join(base_path, '**', '*.i3.zst')
    files = glob.glob(pattern, recursive=True)
    return sorted(files)

def get_relative_path(filepath, base_path):
    rel = os.path.relpath(filepath, base_path)
    return rel.replace(os.sep, '/')

def extract_events(file_list, base_path):
    rows = []
    for filepath in file_list:
        relative_path = get_relative_path(filepath, base_path)
        try:
            f = dataio.I3File(filepath)
            frame_idx = 0
            while f.more():
                frame = f.pop_daq()
                if frame is None:
                    continue
                if 'EventProperties' not in frame:
                    continue
                
                ep = frame['EventProperties']
                props = frame['LeptonInjectorProperties']
                
                rows.append({
                    'file': relative_path,
                    'frame': frame_idx,
                    'TotalEnergy': ep.totalEnergy,
                    'Zenith': ep.zenith,
                    'Azimuth': ep.azimuth,
                    'FinalStateX': ep.finalStateX,
                    'FinalStateY': ep.finalStateY,
                    'FinalType1': str(ep.finalType1),
                    'FinalType2': str(ep.finalType2),
                    'InitialType': str(ep.initialType),
                    'TotalColumnDepth': ep.totalColumnDepth,
                    'ImpactParameter': ep.impactParameter,
                    'InjectionRadius': props.injectionRadius,
                    'EndcapLength': props.endcapLength,
                    'EnergyMin': props.energyMinimum,
                    'EnergyMax': props.energyMaximum,
                    'ZenithMin': props.zenithMinimum,
                    'ZenithMax': props.zenithMaximum,
                    'AzimuthMin': props.azimuthMinimum,
                    'AzimuthMax': props.azimuthMaximum,
                    'PowerlawIndex': props.powerlawIndex,
                    'NEvents': props.events,
                })
                frame_idx += 1
                
        except Exception as e:
            print(f"Hata: {filepath} — {e}")
            continue
    
    return pd.DataFrame(rows)

base_path = "/project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator"
files = get_all_zst_files(base_path)
print(f"Toplam dosya: {len(files)}")

df = extract_events(files, base_path)
print(df.shape)
print(df.head())

Toplam dosya: 88
(7444, 22)
                      file  frame   TotalEnergy    Zenith   Azimuth  \
0  33939527/gen_001.i3.zst      0  30919.790319  1.183465  0.235777   
1  33939527/gen_001.i3.zst      1    420.958879  2.077171  2.882566   
2  33939527/gen_001.i3.zst      2   2953.796868  2.008965  4.649012   
3  33939527/gen_001.i3.zst      3   2710.118250  1.089275  4.854238   
4  33939527/gen_001.i3.zst      4    212.129998  2.711454  0.435339   

   FinalStateX  FinalStateY FinalType1 FinalType2 InitialType  ...  \
0     0.034870     0.224585    MuMinus    Hadrons        NuMu  ...   
1     0.443890     0.539306    MuMinus    Hadrons        NuMu  ...   
2     0.082213     0.544104    MuMinus    Hadrons        NuMu  ...   
3     0.051205     0.312943    MuMinus    Hadrons        NuMu  ...   
4     0.173816     0.953271    MuMinus    Hadrons        NuMu  ...   

   InjectionRadius  EndcapLength  EnergyMin  EnergyMax  ZenithMin  ZenithMax  \
0           1000.0        1100.0      100.0 

In [ ]:
from IPython.display import display
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
display(df)

,file,frame,TotalEnergy,Zenith,Azimuth,FinalStateX,FinalStateY,FinalType1,FinalType2,InitialType,TotalColumnDepth,ImpactParameter,InjectionRadius,EndcapLength,EnergyMin,EnergyMax,ZenithMin,ZenithMax,AzimuthMin,AzimuthMax,PowerlawIndex,NEvents
0,33939527/gen_001.i3.zst,0,30919.790319,1.183465,0.235777,0.034870,0.224585,MuMinus,Hadrons,NuMu,7.075407e+05,107.496771,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
1,33939527/gen_001.i3.zst,1,420.958879,2.077171,2.882566,0.443890,0.539306,MuMinus,Hadrons,NuMu,4.126774e+05,827.318140,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
2,33939527/gen_001.i3.zst,2,2953.796868,2.008965,4.649012,0.082213,0.544104,MuMinus,Hadrons,NuMu,9.381216e+05,532.555700,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
3,33939527/gen_001.i3.zst,3,2710.118250,1.089275,4.854238,0.051205,0.312943,MuMinus,Hadrons,NuMu,7.214660e+05,318.708796,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
4,33939527/gen_001.i3.zst,4,212.129998,2.711454,0.435339,0.173816,0.953271,MuMinus,Hadrons,NuMu,4.269364e+05,786.615967,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7439,34088096/gen_010.i3.zst,74,4369.985976,1.812011,1.234319,0.093155,0.562767,MuPlus,Hadrons,NuMuBar,1.089618e+06,640.378854,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
7440,34088096/gen_010.i3.zst,75,4400.251526,2.452090,4.224942,0.311539,0.331075,MuPlus,Hadrons,NuMuBar,1.092384e+06,795.854420,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
7441,34088096/gen_010.i3.zst,76,600.489653,2.842044,0.971074,0.082019,0.174423,MuPlus,Hadrons,NuMuBar,5.555571e+05,170.421022,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100
7442,34088096/gen_010.i3.zst,77,666.673886,0.362991,2.222807,0.182384,0.416193,MuPlus,Hadrons,NuMuBar,3.963059e+05,727.301460,1000.0,1100.0,100.0,1000000.0,0.0,3.141593,0.0,6.283185,1.5,100


In [ ]:
# Sabit olmasi gereken kolonlar
constant_cols = [
    'InjectionRadius', 'EndcapLength', 'EnergyMin', 'EnergyMax',
    'ZenithMin', 'ZenithMax', 'AzimuthMin', 'AzimuthMax',
    'PowerlawIndex', 'NEvents'
]

print("=== SANITY CHECK ===")
for col in constant_cols:
    unique_vals = df[col].unique()
    status = "✅" if len(unique_vals) == 1 else "❌"
    print(f"{status} {col}: {unique_vals}")

print("\n=== FRAME ===")
print(f"Max frame: {df['frame'].max()}")
print(f"Frame dağılımı:\n{df['frame'].value_counts().sort_index()}")

=== SANITY CHECK ===
✅ InjectionRadius: [1000.]
✅ EndcapLength: [1100.]
✅ EnergyMin: [100.]
✅ EnergyMax: [1000000.]
✅ ZenithMin: [0.]
✅ ZenithMax: [3.14159265]
✅ AzimuthMin: [0.]
✅ AzimuthMax: [6.28318531]
✅ PowerlawIndex: [1.5]
✅ NEvents: [100]

=== FRAME ===
Max frame: 98
Frame dağılımı:
frame
0     88
1     88
2     88
3     88
4     88
5     88
6     88
7     88
8     88
9     88
10    88
11    88
12    88
13    88
14    88
15    88
16    88
17    88
18    88
19    88
20    88
21    88
22    88
23    88
24    88
25    88
26    88
27    88
28    88
29    88
30    88
31    88
32    88
33    88
34    88
35    88
36    88
37    88
38    88
39    88
40    88
41    88
42    88
43    88
44    88
45    88
46    88
47    88
48    88
49    88
50    88
51    88
52    88
53    88
54    88
55    88
56    88
57    88
58    88
59    88
60    88
61    88
62    88
63    88
64    88
65    88
66    88
67    88
68    87
69    86
70    85
71    84
72    81
73    81
74    79
75    78
76    75
77    74
7

In [ ]:
import os
import glob
import numpy as np
import photospline
from icecube import icetray, dataio, dataclasses
import pandas as pd

# Cross section spline'larını yükle
xs_dir = "/project/6008051/pone_simulation/pone_offline/CrossSectionModels/csms_differential_v1.0"

splines = {
    'NuMu':    {
        'sigma_tot': photospline.SplineTable(xs_dir + "/sigma_nu_CC_iso.fits"),
        'sigma_diff': photospline.SplineTable(xs_dir + "/dsdxdy_nu_CC_iso.fits"),
    },
    'NuMuBar': {
        'sigma_tot': photospline.SplineTable(xs_dir + "/sigma_nubar_CC_iso.fits"),
        'sigma_diff': photospline.SplineTable(xs_dir + "/dsdxdy_nubar_CC_iso.fits"),
    },
}

def get_cross_sections(row):
    ptype = row['InitialType']
    log_E = np.log10(row['TotalEnergy'])
    log_x = np.log10(row['FinalStateX'])
    log_y = np.log10(row['FinalStateY'])
    
    sigma_tot  = 10 ** splines[ptype]['sigma_tot'].evaluate_simple([log_E])
    sigma_diff = 10 ** splines[ptype]['sigma_diff'].evaluate_simple([log_E, log_x, log_y])
    
    return sigma_tot, sigma_diff

def get_all_zst_files(base_path):
    pattern = os.path.join(base_path, '**', '*.i3.zst')
    files = glob.glob(pattern, recursive=True)
    return sorted(files)

def get_relative_path(filepath, base_path):
    rel = os.path.relpath(filepath, base_path)
    return rel.replace(os.sep, '/')

def extract_events(file_list, base_path):
    rows = []
    for filepath in file_list:
        relative_path = get_relative_path(filepath, base_path)
        try:
            f = dataio.I3File(filepath)
            frame_idx = 0
            while f.more():
                frame = f.pop_physics()
                if frame is None:
                    continue
                if 'EventProperties' not in frame:
                    continue
                
                ep = frame['EventProperties']
                props = frame['LeptonInjectorProperties']
                
                row = {
                    'file': relative_path,
                    'frame': frame_idx,
                    'TotalEnergy': ep.totalEnergy,
                    'Zenith': ep.zenith,
                    'Azimuth': ep.azimuth,
                    'FinalStateX': ep.finalStateX,
                    'FinalStateY': ep.finalStateY,
                    'FinalType1': str(ep.finalType1),
                    'FinalType2': str(ep.finalType2),
                    'InitialType': str(ep.initialType),
                    'TotalColumnDepth': ep.totalColumnDepth,
                    'ImpactParameter': ep.impactParameter,
                    'InjectionRadius': props.injectionRadius,
                    'EndcapLength': props.endcapLength,
                    'EnergyMin': props.energyMinimum,
                    'EnergyMax': props.energyMaximum,
                    'ZenithMin': props.zenithMinimum,
                    'ZenithMax': props.zenithMaximum,
                    'AzimuthMin': props.azimuthMinimum,
                    'AzimuthMax': props.azimuthMaximum,
                    'PowerlawIndex': props.powerlawIndex,
                }
                
                sigma_tot, sigma_diff = get_cross_sections(row)
                row['sigma_tot']  = sigma_tot
                row['sigma_diff'] = sigma_diff
                
                rows.append(row)
                frame_idx += 1
                
        except Exception as e:
            print(f"Hata: {filepath} — {e}")
            continue
    
    return pd.DataFrame(rows)

base_path = "/project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator"
files = get_all_zst_files(base_path)
print(f"Toplam dosya: {len(files)}")

df = extract_events(files, base_path)
print(df.shape)
print(df.head())

ModuleNotFoundError: No module named 'photospline'

In [ ]:
from icecube import photospline; print('ok')

ImportError: cannot import name 'photospline' from 'icecube' (unknown location)

In [ ]:
from astropy.io import fits

xs_dir = "/project/6008051/pone_simulation/pone_offline/CrossSectionModels/csms_differential_v1.0"
hdul = fits.open(xs_dir + "/sigma_nu_CC_iso.fits")
hdul.info()
print(hdul[0].header)

ModuleNotFoundError: No module named 'astropy'